# 08 · 探索性分析模块（EDA）功能演示

演示数据总览、目标/特征/相关性/稳定性/策略/账龄分析与一键 EDA 报告导出。

In [1]:
import warnings, os
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import hscredit

# 路径约定：从 notebooks/ 目录运行，数据在 ../examples，产物输出到 model_report/
DATA = os.path.join("..", "examples", "hscredit_yyp.xlsx")
if not os.path.exists(DATA):
    DATA = os.path.join("examples", "hscredit_yyp.xlsx")
OUT = "model_report"
os.makedirs(OUT, exist_ok=True)

df = pd.read_excel(DATA)
df["放款时间"] = pd.to_datetime(df["放款时间"])
y = df["FPD"].astype(int)
NUM_FEATURES = ["珊瑚92", "青云24", "衡枢鉴真分老客版", "占信V3", "天创小额网贷分", "近六个月非银多头机构数"]
CAT_FEATURE = "商品类别"
print("数据形状:", df.shape)
print("坏样本率: {:.4f}".format(y.mean()))
df.head()

数据形状: (970, 18)
坏样本率: 0.1402


,客户编号,放款时间,放款金额,商品类别,MOB1,CURRENT_DPD,中智小牛分C3,珊瑚92,极光欺诈分6v1,青云24,占信V3,轻花老客海纳子分V1,天创小额网贷分,近六个月非银多头机构数,手机号近一个月非银多头机构数,身份证近一个月非银多头机构数,衡枢鉴真分老客版,FPD
0,1985945640026276096,2026-02-03,1399,礼包,0,0,NaN,NaN,NaN,656,NaN,NaN,630,51,15,15,0.0242,0
1,1985972188268592896,2026-02-04,1399,礼包,0,0,NaN,NaN,NaN,565,NaN,NaN,583,56,6,18,0.0492,0
2,1986034700861140992,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,708,NaN,NaN,764,68,17,20,0.0546,0
3,1986264852923760896,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,555,NaN,NaN,712,45,15,15,0.0899,0
4,1986265696509906944,2026-01-26,1399,礼包,0,0,NaN,NaN,NaN,581,NaN,NaN,641,67,32,32,0.0678,0


## 1. 数据总览 / 缺失分析 / 特征摘要

In [2]:
import hscredit.core.eda as eda
display(eda.data_info(df))
display(eda.missing_analysis(df))
eda.numeric_summary(df, features=NUM_FEATURES)

,信息项,值
0,样本数（行）,970.0000
1,特征数（列）,18.0000
2,数值型特征,13.0000
3,分类型特征,1.0000
4,日期型特征,1.0000
5,常数特征,0.0000
6,ID特征,3.0000
7,缺失值列数,5.0000
8,总缺失值数,2068.0000
9,内存使用(MB),0.2200


,特征名,缺失数,缺失率(%),非空数,查得率(%)
0,珊瑚92,706,72.7800,264,27.2200
1,中智小牛分C3,663,68.3500,307,31.6500
2,极光欺诈分6v1,662,68.2500,308,31.7500
3,轻花老客海纳子分V1,32,3.3000,938,96.7000
4,占信V3,5,0.5200,965,99.4800
5,客户编号,0,0.0000,970,100.0000
6,衡枢鉴真分老客版,0,0.0000,970,100.0000
7,身份证近一个月非银多头机构数,0,0.0000,970,100.0000
8,手机号近一个月非银多头机构数,0,0.0000,970,100.0000
9,近六个月非银多头机构数,0,0.0000,970,100.0000


,特征名,样本数,均值,标准差,最小值,最大值,中位数,偏度,峰度,1%,5%,95%,99%,零值数,零值率(%),负值数,负值率(%)
0,珊瑚92,264,624.8636,70.3443,440.0000,850.0000,617.0000,0.4687,0.9071,469.2600,505.3500,758.7000,810.7400,0,0.0000,0,0.0000
1,青云24,970,604.3258,64.9345,372.0000,850.0000,603.0000,0.0902,0.5926,452.6900,497.4500,709.5500,756.3100,0,0.0000,0,0.0000
2,衡枢鉴真分老客版,970,0.0946,0.0524,0.0095,0.3076,0.0838,0.9954,0.9490,0.0185,0.0297,0.1931,0.2551,0,0.0000,0,0.0000
3,占信V3,965,573.6383,62.6627,349.0000,762.0000,578.0000,-0.3387,0.1667,418.0000,460.4000,671.0000,702.0800,0,0.0000,0,0.0000
4,天创小额网贷分,970,710.3392,52.5425,497.0000,888.0000,712.0000,-0.1723,0.0675,585.7600,623.0000,792.0000,822.6200,0,0.0000,0,0.0000
5,近六个月非银多头机构数,970,60.8526,12.1311,19.0000,94.0000,61.0000,-0.1715,-0.0802,32.6900,40.0000,81.0000,86.3100,0,0.0000,0,0.0000


## 2. 目标分析：整体坏率 / 多标签坏率 / 维度坏率 / 坏率趋势

In [3]:
display(eda.bad_rate_overall(df, overdue=['MOB1'], dpds=[7, 3, 0]))
display(eda.bad_rate_by_dimension(df, dim_col='商品类别', target_col='FPD'))
eda.bad_rate_trend(df, date_col='放款时间', target_col='FPD', freq='M')

,标签,样本总数,好样本数,坏样本数,逾期率(%)
0,MOB1>7,970,825,145,14.9500
1,MOB1>3,970,808,162,16.7000
2,MOB1>0,970,771,199,20.5200


,维度值,样本数,坏样本数,好样本数,逾期率(%),样本占比(%)
0,珠宝首饰,614,96,518,15.6400,63.3000
1,礼包,189,23,166,12.1700,19.4800
2,手机通讯,141,15,126,10.6400,14.5400
3,电脑数码,20,2,18,10.0000,2.0600
4,家用电器,2,0,2,0.0000,0.2100
5,智能设备,4,0,4,0.0000,0.4100


,时间周期,样本数,好样本数,坏样本数,逾期率(%),环比变化(%)
0,2025-11,356,304,52,14.6100,NaN
1,2025-12,281,240,41,14.5900,-0.0200
2,2026-01,152,124,28,18.4200,3.8300
3,2026-02,181,166,15,8.2900,-10.1300


## 3. 特征分析：IV / WOE / 单调性 / 单变量 AUC

In [4]:
display(eda.batch_iv_analysis(df, NUM_FEATURES, 'FPD'))
display(eda.woe_analysis(df, '衡枢鉴真分老客版', 'FPD'))
print('单调性:', eda.monotonicity_check(df, '衡枢鉴真分老客版', 'FPD'))
print('单变量 AUC:', eda.univariate_auc(df, '衡枢鉴真分老客版', 'FPD'))

,特征名,IV值,预测能力,分箱数
0,珊瑚92,0.3909,强预测能力,10
1,衡枢鉴真分老客版,0.2456,中等预测能力,10
2,占信V3,0.1773,中等预测能力,10
3,青云24,0.1350,中等预测能力,10
4,近六个月非银多头机构数,0.1015,中等预测能力,10
5,天创小额网贷分,0.0555,弱预测能力,10


,分箱,样本数,好样本数,坏样本数,逾期率,WOE值,IV值,LIFT值
0,"[-inf, 0.0370)",97,89,8,0.0825,-0.5956,0.0285,0.5882
1,"[0.0370, 0.0478)",99,89,10,0.1010,-0.3725,0.0124,0.7204
2,"[0.0478, 0.0602)",96,89,7,0.0729,-0.7291,0.0403,0.5201
3,"[0.0602, 0.0719)",96,84,12,0.1250,-0.1323,0.0017,0.8915
4,"[0.0719, 0.0838)",97,80,17,0.1753,0.2648,0.0077,1.2500
5,"[0.0838, 0.0984)",97,86,11,0.1134,-0.2429,0.0054,0.8088
6,"[0.0984, 0.1157)",97,88,9,0.0928,-0.4665,0.0184,0.6618
7,"[0.1157, 0.1342)",98,83,15,0.1531,0.1028,0.0011,1.0917
8,"[0.1342, 0.1679)",95,76,19,0.2000,0.4273,0.0208,1.4265
9,"[0.1679, +inf)",98,70,28,0.2857,0.8973,0.1094,2.0378


单调性: {'特征名': '衡枢鉴真分老客版', '单调性': '单调', '单调方向': '递增', 'Spearman相关系数': 0.7576, 'P值': 0.0111, '说明': '需检查业务含义'}
单变量 AUC: {'特征名': '衡枢鉴真分老客版', 'AUC值': 0.6165, '区分能力': '中等区分能力'}


## 4. 相关性与多重共线性：相关矩阵 / 高相关对 / VIF

In [5]:
display(eda.high_correlation_pairs(df, NUM_FEATURES, threshold=0.5))
eda.vif_analysis(df, NUM_FEATURES)

,信息
0,未发现相关系数>=0.5的特征对


,特征名,VIF值,共线性评级,建议
0,珊瑚92,156.1800,严重共线性,剔除
4,天创小额网贷分,149.5900,严重共线性,剔除
1,青云24,105.6800,严重共线性,剔除
3,占信V3,90.2700,严重共线性,剔除
5,近六个月非银多头机构数,27.6100,严重共线性,剔除
2,衡枢鉴真分老客版,4.7400,无共线性,保留


## 5. 稳定性：PSI / 时序 PSI / 评分漂移

In [6]:
base = df.iloc[:500]; cur = df.iloc[500:]
print('PSI:', eda.psi_analysis(base, cur, '衡枢鉴真分老客版'))
display(eda.feature_drift_report(base, cur, NUM_FEATURES[:3]))
eda.score_drift_report(df['衡枢鉴真分老客版'].fillna(0).values[:500], df['衡枢鉴真分老客版'].fillna(0).values[500:])['分布统计']

PSI: {'特征名': '衡枢鉴真分老客版', 'PSI值': 0.081, '稳定性': '非常稳定', '分箱明细':                  分箱  期望样本数  实际样本数   期望占比   实际占比  PSI贡献
0    [-inf, 0.0370)     45     52 0.0900 0.1106 0.0043
1  [0.0370, 0.0478)     48     51 0.0960 0.1085 0.0015
2  [0.0478, 0.0602)     50     46 0.1000 0.0979 0.0000
3  [0.0602, 0.0719)     46     50 0.0920 0.1064 0.0021
4  [0.0719, 0.0838)     50     47 0.1000 0.1000 0.0000
5  [0.0838, 0.0984)     54     43 0.1080 0.0915 0.0027
6  [0.0984, 0.1157)     44     53 0.0880 0.1128 0.0061
7  [0.1157, 0.1342)     47     51 0.0940 0.1085 0.0021
8  [0.1342, 0.1679)     47     48 0.0940 0.1021 0.0007
9    [0.1679, +inf)     69     29 0.1380 0.0617 0.0614}


,特征名,PSI,偏移等级,均值变化(%),缺失率变化(%),基准均值,目标均值,基准缺失率(%),目标缺失率(%)
0,青云24,0.3425,不稳定,4.7400,0.0000,590.7500,618.7681,0.0000,0.0000
1,衡枢鉴真分老客版,0.0810,非常稳定,-10.1900,0.0000,0.0995,0.0894,0.0000,0.0000
2,珊瑚92,NaN,数据不足,NaN,NaN,NaN,NaN,NaN,NaN


,样本数,均值,中位数,标准差,P10,P90,数据集
0,500,0.0995,0.0873,0.0564,0.0392,0.1805,基准
1,470,0.0894,0.0807,0.0473,0.0358,0.1526,目标


## 6. 策略分析：通过率-坏率权衡 / 评分策略模拟

In [7]:
score = df['衡枢鉴真分老客版'].fillna(df['衡枢鉴真分老客版'].median()).values
display(eda.approval_badrate_tradeoff(y.values, score, n_points=20))
eda.score_strategy_simulation(df.assign(score=score), 'score', 'FPD', thresholds=[550, 600, 650], amount_col='放款金额')

,评分阈值,通过率(%),拒绝率(%),通过人数,拒绝人数,通过人群坏率(%),拒绝人群坏率(%),通过人群坏样本数,拒绝人群坏样本数
0,0.1979,4.8500,95.1500,47,923,34.0426,13.0011,16,120
1,0.1706,9.5900,90.4100,93,877,29.0323,12.4287,27,109
2,0.1527,14.3300,85.6700,139,831,28.0576,11.6727,39,97
3,0.1368,19.0700,80.9300,185,785,24.8649,11.4650,46,90
4,0.1257,23.8100,76.1900,231,739,22.5108,11.3667,52,84
5,0.1176,28.5600,71.4400,277,693,21.6606,10.9668,60,76
6,0.1107,33.4000,66.6000,324,646,19.4444,11.3003,63,73
7,0.1015,38.1400,61.8600,370,600,18.3784,11.3333,68,68
8,0.0931,42.8900,57.1100,416,554,17.7885,11.1913,74,62
9,0.0873,47.6300,52.3700,462,508,17.5325,10.8268,81,55


,评分阈值,通过量(笔),通过率(%),通过人群坏率(%),捕获坏样本数,拒绝量(笔),拒绝率(%),拒绝坏样本数,坏样本拦截率(%),通过金额,通过人群坏账金额,"坏账率(金额,%)"
0,550,0,0.0000,NaN,0,970,100.0000,136,100.0000,0.0000,0.0000,NaN
1,600,0,0.0000,NaN,0,970,100.0000,136,100.0000,0.0000,0.0000,NaN
2,650,0,0.0000,NaN,0,970,100.0000,136,100.0000,0.0000,0.0000,NaN


## 7. 账龄 / 滚动率分析（vintage / roll rate）

In [8]:
vdf = df.copy()
vdf['vintage'] = vdf['放款时间'].dt.to_period('M').astype(str)
vdf['mob'] = np.random.RandomState(0).randint(1, 7, len(vdf))
display(eda.vintage_summary(vdf, 'vintage', 'mob', 'FPD'))
eda.roll_rate_analysis(vdf.assign(dpd0=np.random.RandomState(1).randint(0,30,len(vdf)),
                                  dpd1=np.random.RandomState(2).randint(0,40,len(vdf))),
                       overdue_cols=['dpd0','dpd1'])

,Vintage批次,总开户数,MOB0坏账率(%),MOB1坏账率(%),MOB2坏账率(%),MOB3坏账率(%),MOB4坏账率(%),MOB5坏账率(%),MOB6坏账率(%),MOB7坏账率(%),MOB8坏账率(%),MOB9坏账率(%),MOB10坏账率(%),MOB11坏账率(%),MOB12坏账率(%)
0,2025-11,356,NaN,1.6900,4.2100,6.4600,8.4300,11.2400,14.6100,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-12,281,NaN,1.7800,3.5600,6.0500,8.5400,11.0300,14.5900,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-01,152,NaN,1.3200,3.9500,8.5500,11.1800,14.4700,18.4200,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-02,181,NaN,1.1000,1.6600,3.3100,4.4200,6.0800,8.2900,NaN,NaN,NaN,NaN,NaN,NaN


,期数,状态,户数,占比(%)
0,状态1,0,47,4.8500
1,状态1,7,45,4.6400
2,状态1,28,40,4.1200
3,状态1,22,39,4.0200
4,状态1,13,37,3.8100
...,...,...,...,...
65,状态2,14,17,1.7500
66,状态2,13,16,1.6500
67,状态2,4,15,1.5500
68,状态2,3,12,1.2400


## 8. 一键 EDA 报告并导出 Excel

In [9]:
report = eda.generate_report(df, target='FPD', features=NUM_FEATURES, date_col='放款时间')
eda.export_report_to_excel(report, f"{OUT}/08_eda_report.xlsx")
print('已导出 EDA 报告，章节数:', len(report) if hasattr(report,'__len__') else 'n/a')

已导出 EDA 报告，章节数: 7
